# Colab setup: Google Drive workspace + data

Run top to bottom at the start of every Colab session.

The data (`data/processed/`, ...) comes pre-built as a single archive on Drive,
`MyDrive/<DRIVE_DIR>/data.zip`. It is copied to local disk and extracted into the repo's `data/`
folder (local disk is much faster than Drive for many small files). No download or Phase-1
pipeline run is needed.

| Where | What |
|---|---|
| `MyDrive/<DRIVE_DIR>/data.zip` | pre-built data archive (shared by a teammate) |
| `MyDrive/<DRIVE_DIR>/outputs/` | logs, training runs (persist across sessions) |
| `/content/deep-learning-cse365/` | repo + extracted `data/` |

In [ ]:
# ---- Settings ----
DRIVE_DIR = "cse365_pothole"   # personal folder under MyDrive (contains data.zip)
REPO_URL = "https://github.com/queso-gato1355/deep-learning-cse365.git"
BRANCH = "main"

In [ ]:
# 1. Mount Drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_DIR
DATA_ZIP = DRIVE_ROOT / "data.zip"
if not DATA_ZIP.exists():
    raise FileNotFoundError(f"{DATA_ZIP} not found; check DRIVE_DIR.")
for sub in ["outputs/logs", "outputs/runs"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print("Drive workspace:", DRIVE_ROOT)

In [ ]:
# 2. Clone (or update) the repo on local disk
import subprocess

REPO = Path("/content/deep-learning-cse365")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
%cd {REPO}
!git log --oneline -3

In [ ]:
# 3. Dependencies not preinstalled on Colab
!pip install -q albumentations==2.0.8 ruff==0.17.3

In [ ]:
# 4. Link persistent Drive folders for logs and training runs
import os
import shutil

def link(target: Path, link_path: Path) -> None:
    """Replace ``link_path`` with a symlink to ``target``."""
    link_path.parent.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        link_path.unlink()
    elif link_path.exists():
        shutil.rmtree(link_path)
    os.symlink(target, link_path)

link(DRIVE_ROOT / "outputs/logs", REPO / "outputs/logs")
link(DRIVE_ROOT / "outputs/runs", REPO / "outputs/runs")
!ls -la outputs

In [ ]:
# 5. Copy data.zip to local disk and extract it into the repo's data/ folder.
#    Works whether the archive has a top-level "data/" folder or not.
#    Skipped if it was already extracted in this session (marker file).
import zipfile

MARKER = REPO / "data/.unzipped"
if MARKER.exists():
    print("data.zip already extracted in this session: skipped.")
else:
    local_zip = Path("/content/data.zip")
    print(f"Copying {DATA_ZIP} ({DATA_ZIP.stat().st_size / 1e9:.2f} GB) to local disk ...")
    shutil.copyfile(DATA_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as zf:
        names = [n for n in zf.namelist() if not n.startswith("__MACOSX/")]
    dest = REPO if all(n.startswith("data/") for n in names) else REPO / "data"
    dest.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {len(names)} entries into {dest} ...")
    subprocess.run(["unzip", "-q", "-o", str(local_zip), "-x", "__MACOSX/*", "-d", str(dest)], check=True)
    local_zip.unlink()
    MARKER.touch()
    print("Done.")
!du -sh data/* 2>/dev/null
!git status --short data/processed/splits

## Disk check

`git status` above should print nothing: the extracted split lists must match the committed ones.

In [ ]:
!df -h /content /content/drive | cat